# mAbs Animal-Study Pipeline (Colab)

End-to-end build of a structured table of mAb animal-study findings
from a PubMed query, plus per-stage eval runners. Everything — every
stage, every eval, every input (query, screening criteria, models,
extraction schema) — lives in this notebook and is editable below.

**Order:** Stage 1 -> Eval 1 -> Stage 2 -> Eval 2 -> ... -> Stage 6 -> Eval 6 -> Summary.

**What gets written to disk:** each stage writes under
`stage_NN/data/` (production artifacts) and `stage_NN/eval/` (graded
artifacts) in the notebook's working directory. Inspect those files
between stages if you want.


## 1. Install dependencies

`openai` for the LLM stages and evals; `pandas` for result tables;
the `liteparse` Node CLI for PDF -> text in Stage 4.


In [ ]:
%pip install -q openai>=1.40 pandas
!npm i -g @llamaindex/liteparse 2>&1 | tail -n 3


## 2. Set `OPENAI_API_KEY`

Stages 2 + 5 and the LLM evals (eval 1, 2, 5) need OpenAI. Stage 2
falls back to a regex emulator without a key; everything else either
runs offline or skips cleanly.

In Colab: store the key under Secrets (key icon in the left sidebar)
as `OPENAI_API_KEY`, then run this cell. Outside Colab: set the env
var before launching Jupyter.


In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore
    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
        print("OPENAI_API_KEY loaded from Colab Secrets.")
    else:
        print("WARN: OPENAI_API_KEY not set in Colab Secrets — LLM stages will be skipped.")
except ImportError:
    if os.environ.get("OPENAI_API_KEY"):
        print("OPENAI_API_KEY found in environment.")
    else:
        print("WARN: OPENAI_API_KEY not set — LLM stages will be skipped.")


## 3. Configuration

All editable inputs live here. Change values, then **Run all** (or
re-run the cells below). The rest of the notebook reads these
variables directly.


In [ ]:
# ---------------------------------------------------------------------------
# Stage 1 — PubMed search
# ---------------------------------------------------------------------------

# PubMed esearch query. The workshop's canonical query targets
# mAb development + in-vivo arms + 2025-2026 publication date.
QUERY = (
    '("monoclonal antibody"[Title/Abstract] OR mAb[Title/Abstract] '
    'OR "monoclonal antibodies"[Title/Abstract]) '
    'AND (pharmacokinetic*[Title/Abstract] OR toxicology[Title/Abstract] '
    'OR toxicity[Title/Abstract] OR immunogenicity[Title/Abstract] '
    'OR biodistribution[Title/Abstract]) '
    'AND (cynomolgus[Title/Abstract] OR "non-human primate"[Title/Abstract] '
    'OR NHP[Title/Abstract] OR mouse[Title/Abstract] '
    'OR rat[Title/Abstract] OR rodent[Title/Abstract]) '
    'AND ("2025"[Date - Publication] : "2026"[Date - Publication])'
)

N = 30                                     # max PMIDs from esearch
TOOL  = "ar-bic-2026-workshop"             # NCBI identification
EMAIL = "workshop@example.org"

# ---------------------------------------------------------------------------
# Stage 2 — screen abstracts
# ---------------------------------------------------------------------------

SCREENER_MODEL = "gpt-5.4-nano"
SCREEN_SLEEP_SECONDS = 1.0

CRITERIA = """\
Include: primary research papers (2025-2026) reporting at least one
in-vivo mammalian study arm in support of monoclonal antibody (mAb)
development. Eligible study arms include:
  - pharmacokinetics (PK) or toxicokinetics
  - single-dose or repeat-dose toxicology
  - immunogenicity / anti-drug antibody (ADA) assessment
  - tissue biodistribution
  - tissue cross-reactivity confirmed in vivo

Eligible species: mouse, rat, cynomolgus monkey, rhesus monkey, dog,
rabbit, minipig.

Exclude:
  - reviews, meta-analyses, perspectives, commentaries, editorials
  - papers reporting only in-vitro binding, cell-line, or PBMC work
    with no in-vivo arm
  - veterinary mAb studies (animal as patient, not as preclinical model)
  - discovery-stage efficacy-only papers using mouse xenograft tumor
    models with no PK/tox/immunogenicity arm (mouse xenograft efficacy
    alone is not the reducible step we are studying)
  - mAb-conjugate papers where the conjugate (radioligand, toxin) is
    the primary subject and the antibody is incidental
  - case reports of mAb adverse events in patients
"""

# ---------------------------------------------------------------------------
# Stage 5 — structured extraction
# ---------------------------------------------------------------------------

EXTRACTOR_MODEL = "gpt-5.4-nano"
EXTRACT_SLEEP_SECONDS = 1.0

# Extraction contract.
SCHEMA = """\
{
  "pmid": "string",
  "source_type": "fulltext | abstract-only",
  "first_author": "string",
  "year": "integer",
  "mab_name": "string | null",
  "target": "string",
  "format": "IgG1 | IgG2 | IgG3 | IgG4 | bispecific | ADC | Fab | Fc-fusion | other",
  "development_stage": "discovery | lead optimization | IND-enabling | clinical translation | post-approval",
  "regulatory_context": "none-stated | IND-supporting | BLA-supporting | post-marketing",
  "threeRs_mentioned": "boolean",
  "author_reduction_recommendation": "string | null",
  "animal_arms": [
    {
      "species": "mouse | rat | cynomolgus | rhesus | dog | rabbit | minipig | other",
      "n_animals": "integer | null",
      "study_type": "PK | single-dose tox | repeat-dose tox | immunogenicity | biodistribution | efficacy | TCR",
      "duration_days": "integer | null",
      "species_justification": "pharmacological relevance | regulatory expectation | historical precedent | not stated",
      "cross_reactivity_evidence": "in-vitro binding shown | sequence homology only | not addressed",
      "endpoints_unique_to_animal": "string | null",
      "concurrent_nam": "string | null"
    }
  ],
  "nams_discussed": [
    {
      "method": "string",
      "context": "future work | limitation discussion | literature comparison"
    }
  ]
}
"""

# ---------------------------------------------------------------------------
# Eval — LLM grader / generator models
# ---------------------------------------------------------------------------

EVAL_MODEL     = "gpt-5.4-nano"            # grader (cheap, dominates cost)
EVAL_GEN_MODEL = "gpt-5.4-mini"            # question generator (smarter)
EVAL_05_MODEL  = "gpt-5.4-mini"            # eval_05_llm grader (mini-on-mini)

EVAL_02_N_QUESTIONS = 5                    # T/F per record in eval_02_llm
EVAL_05_N_QUESTIONS = 10                   # T/F per paper in eval_05_llm
EVAL_05_N_PAPERS    = 2                    # papers to grade in eval_05_llm

print("Config loaded.")
print(f"  Query: {QUERY[:80]}...")
print(f"  N PMIDs: {N}")
print(f"  Screener: {SCREENER_MODEL}  |  Extractor: {EXTRACTOR_MODEL}")
print(f"  Eval grader: {EVAL_MODEL}  |  Eval generator: {EVAL_GEN_MODEL}")


## 4. Helpers

Small utilities used across stages: section headers, score
bookkeeping, and a DataFrame display helper.


In [ ]:
import json
import os
from contextlib import contextmanager

try:
    import pandas as pd
    from IPython.display import display, Markdown
    _HAS_DISPLAY = True
except ImportError:
    _HAS_DISPLAY = False


def section(title, sub=""):
    """Print a visible section header in cell output."""
    bar = "=" * 72
    print(f"\n{bar}\n{title}")
    if sub:
        print(sub)
    print(bar)


@contextmanager
def step(label):
    """Print a 'running …' / 'done' pair so progress is visible."""
    print(f"\n[{label}] running ...")
    try:
        yield
    finally:
        print(f"[{label}] done.")


def write_score(stage, key, passed, total):
    """Merge {passed, total, percent} under `key` into stage/eval/score.json."""
    os.makedirs(f"{stage}/eval", exist_ok=True)
    path = f"{stage}/eval/score.json"
    data = {}
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
    pct = round(100.0 * passed / total, 1) if total else 0.0
    data[key] = {"passed": passed, "total": total, "percent": pct}
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"  score: {stage} {key} = {passed}/{total} ({pct}%)")


def show_df(df, caption=None, max_rows=12):
    """Render a DataFrame nicely if IPython is available; otherwise print."""
    if _HAS_DISPLAY:
        if caption:
            display(Markdown(f"**{caption}**"))
        display(df.head(max_rows))
    else:
        if caption:
            print(caption)
        print(df.head(max_rows).to_string(index=False))


def need_openai():
    """Return True if OPENAI_API_KEY is set; print a skip notice otherwise."""
    if os.environ.get("OPENAI_API_KEY"):
        return True
    print("  SKIP: no OPENAI_API_KEY in environment.")
    return False


## Stage 1 — PubMed search + metadata

Two NCBI E-utilities calls: `esearch` (query → PMIDs, capped at `N`)
then `efetch` (PMIDs → full metadata: title, abstract, authors,
journal, year, pub_types). Uses `.itertext()` so inline XML children
(`<i>`, `<sub>`, `<sup>`) survive — a common silent-truncation bug
when reading PubMed XML naively.

Writes `stage_01/data/pmids.json` and displays the first records.


### Your task — write Stage 1

Implement Stage 1 in the empty code cell at the bottom of this
section.

To use Gemini: open Colab's **Gemini** panel (sparkle icon, top-right
of the toolbar). Copy the next cell's content (cell menu -> *Copy cell
content*, or double-click the cell, select-all, copy) and paste it
into Gemini as your prompt. Review the code Gemini gives you, paste
it into the empty cell below, then run.

Once it runs cleanly, run the eval cells below to validate your work.


Goal: complete a Python cell that fetches PubMed metadata via NCBI E-utilities.

Already defined in earlier cells: QUERY (str), N (int), TOOL (str),
EMAIL (str). Helpers `section(title, sub)`, `step(label)` (context
manager), `show_df(df, caption=...)`.

Use only Python stdlib (`json`, `os`, `urllib.parse`, `urllib.request`,
`xml.etree.ElementTree`) plus `pandas`. Set
`HEADERS = {"User-Agent": "ar-bic-2026/0.1"}` for every request.

Steps:
1. Call NCBI esearch
   `https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi`
   with params `db=pubmed, term=QUERY, retmax=N, retmode=json,
   sort=date, tool=TOOL, email=EMAIL`. Get the list of PMIDs from
   `esearchresult.idlist`.

2. Call NCBI efetch
   `https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi`
   with `db=pubmed, id=<comma-joined PMIDs>, retmode=xml`. Parse each
   `<PubmedArticle>` and extract:
   - `pmid` (str)
   - `title` (str)
   - `abstract` (str): concatenation of every `<AbstractText>` child;
     each can have a `Label` attribute (BACKGROUND/METHODS/RESULTS/
     CONCLUSIONS) - when set, prefix the text with `"<Label>: "`.
   - `authors` (list[str]): walk `<AuthorList>/<Author>`. Prefer
     `<CollectiveName>`, else `"<LastName> <ForeName>"` (fall back to
     `<Initials>` if no `<ForeName>`).
   - `first_author` (str): `authors[0]` if any, else `""`.
   - `journal` (str): `<Journal>/<Title>` else `<ISOAbbreviation>`.
   - `year` (int | None): from `<PubDate>/<Year>`, or first 4 chars of
     `<MedlineDate>` if Year missing.
   - `pub_types` (list[str]): every `<PublicationType>` text.

   Critical: use `"".join(el.itertext()).strip()` (NOT `el.text`) for
   any element whose value can contain inline children like `<i>`,
   `<sub>`, `<sup>`. Using `.text` silently truncates abstracts at
   the first inline child.

3. Re-sort `records` to match the esearch PMID order (efetch may
   reorder).

4. Create `stage_01/data/` if missing. Write
   `stage_01/data/pmids.json` with shape:
   `{"query": QUERY, "n_requested": N, "pmids": [...], "records": [...]}`
   (indent=2).

5. Assertions: pmids non-empty; every pmid is `.isdigit()`; every
   record has non-empty `pmid` and `title`; `len(records) == len(pmids)`.

6. Print a one-line OK summary. Then build a pandas DataFrame with
   columns `pmid, year, first_author, journal[:40], title[:80]` and
   pass it to `show_df(df, caption=...)`.

Open with `section("Stage 1 - PubMed search", ...)`. Use
`step("esearch")` and `step("efetch metadata")` for sub-phases.


In [ ]:
# Your Stage 1 implementation goes here.
# Paste Gemini's output (or your own code) into this cell.


### Eval — Stage 1

Two halves:

1. **Script-only** (offline) — PMID-list shape + per-record metadata
   completeness, including an XML-tag-leak regex that catches the
   `.text`-vs-`.itertext()` truncation bug.
2. **LLM T/F** (needs `OPENAI_API_KEY`) — flag records whose title or
   abstract length falls below a threshold, then ask the model
   whether each flagged field is **complete** (not cut off mid-word
   or mid-sentence). Topic relevance is deliberately ignored — Stage
   2 handles that.


In [ ]:
import datetime
import json
import os
import re

section("Eval Stage 1 — script checks", "deterministic re-checks of pmids.json")

XML_TAG_LEAK = re.compile(r"</?[a-zA-Z]")
STAGE = "stage_01"
os.makedirs(f"{STAGE}/eval", exist_ok=True)

REQUIRED_FIELDS = {"pmid", "title", "abstract", "authors", "first_author",
                   "journal", "year", "pub_types"}
THIS_YEAR = datetime.date.today().year

with open(f"{STAGE}/data/pmids.json") as f:
    doc = json.load(f)
pmids = doc["pmids"]
records = doc.get("records") or []

script_checks = {
    "pmids_non_empty": bool(pmids),
    "all_numeric": all(p.isdigit() for p in pmids),
    "no_duplicates": len(pmids) == len(set(pmids)),
    "respects_cap": len(pmids) <= doc.get("n_requested", len(pmids)),
    "records_match_pmids": [r["pmid"] for r in records] == pmids,
}

per_record_issues = []
n_fields_present = n_title_ok = n_abstract_long = n_abstract_no_xml_leak = 0
n_authors_non_empty = n_journal_ok = n_year_ok = n_pubtypes_ok = 0

for r in records:
    issues = []
    if set(r.keys()) >= REQUIRED_FIELDS: n_fields_present += 1
    else: issues.append(f"missing keys: {REQUIRED_FIELDS - set(r.keys())}")
    if r.get("title") and len(r["title"]) > 10: n_title_ok += 1
    else: issues.append("title missing or < 10 chars")
    abstract = r.get("abstract") or ""
    if len(abstract) > 200: n_abstract_long += 1
    if not XML_TAG_LEAK.search(abstract): n_abstract_no_xml_leak += 1
    else: issues.append("abstract contains XML-tag-shaped leakage")
    if r.get("authors"): n_authors_non_empty += 1
    if r.get("journal"): n_journal_ok += 1
    year = r.get("year")
    if isinstance(year, int) and 1990 <= year <= THIS_YEAR + 1: n_year_ok += 1
    else: issues.append(f"year out of range: {year!r}")
    if r.get("pub_types"): n_pubtypes_ok += 1
    else: issues.append("pub_types empty")
    if issues:
        per_record_issues.append({"pmid": r.get("pmid"), "issues": issues})

n = len(records)
script_checks.update({
    "all_records_have_required_fields": n_fields_present == n,
    "all_titles_substantive": n_title_ok == n,
    "no_xml_tag_leakage_in_abstracts": n_abstract_no_xml_leak == n,
    "most_abstracts_full_length": (n_abstract_long / n) >= 0.6 if n else True,
    "most_records_have_authors": (n_authors_non_empty / n) >= 0.9 if n else True,
    "all_journals_named": n_journal_ok == n,
    "all_years_in_range": n_year_ok == n,
    "all_records_have_pub_types": n_pubtypes_ok == n,
})

for k, v in script_checks.items():
    print(f"  {'OK  ' if v else 'FAIL'}  {k}")
if per_record_issues:
    print(f"\nPer-record issues ({len(per_record_issues)}):")
    for it in per_record_issues[:5]:
        print(f"  {it['pmid']}: {'; '.join(it['issues'])}")

with open(f"{STAGE}/eval/eval_script.json", "w") as f:
    json.dump({"script": script_checks, "per_record_issues": per_record_issues}, f, indent=2)

n_total = len(script_checks)
n_passed = sum(1 for v in script_checks.values() if v)
write_score(STAGE, "script", n_passed, n_total)


In [ ]:
# Eval 1 — LLM truncation check on records below length thresholds
section("Eval Stage 1 — LLM T/F",
        f"grader={EVAL_MODEL}, truncation check on records below length thresholds")

if need_openai():
    from openai import OpenAI
    client = OpenAI()
    STAGE = "stage_01"
    os.makedirs(f"{STAGE}/eval", exist_ok=True)

    # Records whose title/abstract is shorter than these get an LLM check
    # for "is this complete, or is it cut off?"
    TITLE_LEN_THRESHOLD = 30
    ABSTRACT_LEN_THRESHOLD = 200

    def grade(question, context):
        prompt = (
            "Answer with strictly TRUE or FALSE based only on the context. "
            "No scoring, no 'partial', no hedging, no explanation. "
            'Return JSON: {"answer": "TRUE" | "FALSE"}.\n\n'
            f"QUESTION:\n{question}\n\nCONTEXT:\n{context}"
        )
        resp = client.chat.completions.create(
            model=EVAL_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0, response_format={"type": "json_object"},
        )
        return json.loads(resp.choices[0].message.content).get("answer", "").upper() == "TRUE"

    with open(f"{STAGE}/data/pmids.json") as f:
        records = json.load(f).get("records") or []

    QUESTIONS = {
        "title": (
            "Does this string look like a COMPLETE PubMed article title — "
            "not cut off mid-word or mid-phrase, not ending with an "
            "ellipsis or '[truncated]'? An intentionally short but "
            "well-formed title is TRUE."
        ),
        "abstract": (
            "Does this string look like a COMPLETE PubMed abstract — "
            "not cut off mid-sentence, not marked with '...' or "
            "'[truncated]'? An intentionally short but well-formed "
            "abstract is TRUE. An empty abstract is FALSE."
        ),
    }

    # Flag records whose title/abstract length is below threshold
    flagged = []
    for rec in records:
        flags = []
        title = rec.get("title", "") or ""
        abstract = rec.get("abstract", "") or ""
        if len(title) < TITLE_LEN_THRESHOLD:
            flags.append("title")
        if len(abstract) < ABSTRACT_LEN_THRESHOLD:
            flags.append("abstract")
        if flags:
            flagged.append((rec, flags))

    print(f"  {len(flagged)}/{len(records)} records below thresholds "
          f"(title<{TITLE_LEN_THRESHOLD}, abstract<{ABSTRACT_LEN_THRESHOLD})")

    items = []
    for i, (rec, flags) in enumerate(flagged, 1):
        print(f"  [{i}/{len(flagged)}] truncation check  pmid={rec.get('pmid')}  fields={flags}")
        checks = []
        for field in flags:
            value = rec.get(field, "") or ""
            ctx = f"FIELD: {field}\nVALUE:\n{value}"
            checks.append({"field": field, "q": QUESTIONS[field],
                           "answer": grade(QUESTIONS[field], ctx)})
        items.append({"pmid": rec.get("pmid"), "flags": flags, "checks": checks})

    with open(f"{STAGE}/eval/eval_llm.json", "w") as f:
        json.dump({"ai": items, "thresholds": {
            "title": TITLE_LEN_THRESHOLD,
            "abstract": ABSTRACT_LEN_THRESHOLD,
        }}, f, indent=2)

    trues = sum(1 for it in items for c in it["checks"] if c["answer"])
    total = sum(len(it["checks"]) for it in items)
    if total:
        print(f"\nAI T/F: {trues}/{total} TRUE (complete) across "
              f"{len(flagged)} flagged records")
    else:
        print("\nNo records below thresholds — nothing to check.")
        # No flagged records counts as a pass
        trues, total = 1, 1
    write_score(STAGE, "llm", trues, total)
